# Milestone 2 — Chunking, Embedding, Storing (Data Indexing)

**Ziel:** Aus der gespeicherten Transcript (M1) durchsuchbare Chunks in einer Vector-DB machen.

**Input:** `data/transcripts/E7W4OQfJWdw.json` (2502 rohe Segmente, aus M1)
**Output dieses Notebooks:** Chunks + Embeddings, gespeichert in Chroma


In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")
client = OpenAI()

print("API-Key gefunden:", client.api_key is not None)

API-Key gefunden: True


In [1]:
import os
import json

# Transcript aus M1 laden -- wir müssen nichts neu von YouTube abrufen
with open("../data/transcripts/E7W4OQfJWdw.json", "r", encoding="utf-8") as f:
    transcript_data = json.load(f)

segments = transcript_data["segments"]
video_id = transcript_data["video_id"]

print(f"Video-ID: {video_id}")
print(f"Anzahl roher Segmente: {len(segments)}")
print(f"Erstes Segment: {segments[0]}")

Video-ID: E7W4OQfJWdw
Anzahl roher Segmente: 2502
Erstes Segment: {'start': 0.36, 'end': 2.28, 'text': '- Welcome to the Huberman Lab Podcast,'}


## Chunking: Segmente zu 30-Sekunden-Fenstern zusammenfassen

Wir gruppieren aufeinanderfolgende Segmente, bis das Zeitfenster 30 Sekunden überschreitet. Jeder Chunk behält `start` (Beginn des ersten Segments) und `end` (Ende des letzten Segments im Chunk).


In [2]:
def chunk_by_time(segments: list, window_seconds: float = 30.0) -> list:
    chunks = []
    current_chunk_segments = []
    current_chunk_start = None

    for seg in segments:
        if current_chunk_start is None:
            current_chunk_start = seg["start"]

        current_chunk_segments.append(seg)

        # Prüfen: Ist das aktuelle Zeitfenster (von Chunk-Start bis jetzt) schon 30s+?
        elapsed = seg["end"] - current_chunk_start
        if elapsed >= window_seconds:
            # Chunk abschließen: Texte zusammenfügen, Start/Ende festhalten
            chunk_text = " ".join(s["text"] for s in current_chunk_segments)
            chunks.append({
                "start": current_chunk_start,
                "end": current_chunk_segments[-1]["end"],
                "text": chunk_text,
            })
            # Neuen Chunk vorbereiten
            current_chunk_segments = []
            current_chunk_start = None

    # Letzten, evtl. unvollständigen Chunk auch noch hinzufügen (falls Segmente übrig sind)
    if current_chunk_segments:
        chunk_text = " ".join(s["text"] for s in current_chunk_segments)
        chunks.append({
            "start": current_chunk_start,
            "end": current_chunk_segments[-1]["end"],
            "text": chunk_text,
        })

    return chunks


chunks = chunk_by_time(segments)
print(f"Aus {len(segments)} Segmenten wurden {len(chunks)} Chunks")
print(f"\nBeispiel-Chunk:\n{chunks[0]}")

Aus 2502 Segmenten wurden 194 Chunks

Beispiel-Chunk:
{'start': 0.36, 'end': 31.049999999999997, 'text': "- Welcome to the Huberman Lab Podcast, where we discuss science and science-based tools for everyday life. [upbeat rock music] I'm Andrew Huberman, and I'm a professor of\nneurobiology and ophthalmology at Stanford School of Medicine. Today, we are talking all\nabout food and the brain. We are going to talk about foods that are good for your\nbrain in terms of focus, in terms of brain health generally, and the longevity of your brain, your ability to maintain cognition and clear thinking over time."}


## Cleaning: Nicht-Sprache-Einträge entfernen

Whisper/YouTube markieren Nicht-Sprache-Momente (Musik, Applaus, etc.) mit eckigen Klammern wie `[upbeat rock music]`. Die filtern wir jetzt raus, bevor wir chunken.


In [3]:
import re

def clean_segments(segments: list) -> list:
    cleaned = []
    for seg in segments:
        text = seg["text"]

        # Entfernt alles in eckigen Klammern, z.B. [upbeat rock music], [applause]
        text = re.sub(r"\[.*?\]", "", text)

        # Überflüssige Leerzeichen glattziehen, die durchs Entfernen entstehen
        text = " ".join(text.split())

        # Leere Segmente (die NUR aus einer Klammer bestanden) ganz weglassen
        if text:
            cleaned.append({**seg, "text": text})

    return cleaned


cleaned_segments = clean_segments(segments)
print(f"Vorher: {len(segments)} Segmente")
print(f"Nachher: {len(cleaned_segments)} Segmente (leere entfernt)")

# Jetzt NEU chunken, mit den bereinigten Segmenten
chunks = chunk_by_time(cleaned_segments)
print(f"\nDaraus: {len(chunks)} Chunks")
print(f"\nBeispiel-Chunk:\n{chunks[0]}")

Vorher: 2502 Segmente
Nachher: 2500 Segmente (leere entfernt)

Daraus: 194 Chunks

Beispiel-Chunk:
{'start': 0.36, 'end': 31.049999999999997, 'text': "- Welcome to the Huberman Lab Podcast, where we discuss science and science-based tools for everyday life. I'm Andrew Huberman, and I'm a professor of neurobiology and ophthalmology at Stanford School of Medicine. Today, we are talking all about food and the brain. We are going to talk about foods that are good for your brain in terms of focus, in terms of brain health generally, and the longevity of your brain, your ability to maintain cognition and clear thinking over time."}


## Metadata anreichern

Bevor wir embedden, hängen wir an jeden Chunk die Metadata, die wir für Filterung/Zitation später brauchen: `video_id`, `title`, `source`.


In [4]:
def add_metadata(chunks: list, video_id: str, title: str) -> list:
    enriched = []
    for i, chunk in enumerate(chunks):
        enriched.append({
            **chunk,
            "chunk_id": f"{video_id}_{i}",   # eindeutige ID pro Chunk, z.B. "E7W4OQfJWdw_0"
            "video_id": video_id,
            "title": title,
        })
    return enriched


chunks_with_metadata = add_metadata(
    chunks,
    video_id=video_id,
    title="Nutrients For Brain Health & Performance | Huberman Lab Podcast #42",
)

print(chunks_with_metadata[0])

{'start': 0.36, 'end': 31.049999999999997, 'text': "- Welcome to the Huberman Lab Podcast, where we discuss science and science-based tools for everyday life. I'm Andrew Huberman, and I'm a professor of neurobiology and ophthalmology at Stanford School of Medicine. Today, we are talking all about food and the brain. We are going to talk about foods that are good for your brain in terms of focus, in terms of brain health generally, and the longevity of your brain, your ability to maintain cognition and clear thinking over time.", 'chunk_id': 'E7W4OQfJWdw_0', 'video_id': 'E7W4OQfJWdw', 'title': 'Nutrients For Brain Health & Performance | Huberman Lab Podcast #42'}


## Embedding-Modell wählen

Wir nutzen OpenAIs Embedding-API — passt zu unserer Deployment-Entscheidung (API-basiert statt selbst-gehostet, kein großes Modell im Deployment-Container).

**Modellwahl:** `text-embedding-3-small` — güntiger und kleiner als `text-embedding-3-large`, für unseren MVP-Scope (ein Video, 194 Chunks) völlig ausreichend.


In [7]:
def embed_texts(texts: list) -> list:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts,
    )
    # response.data ist eine Liste von Embedding-Objekten, in derselben Reihenfolge wie die Input-Texte
    return [item.embedding for item in response.data]


# Alle Chunk-Texte auf einmal einbetten (effizienter als einzeln)
all_texts = [chunk["text"] for chunk in chunks_with_metadata]
embeddings = embed_texts(all_texts)

print(f"Anzahl Embeddings: {len(embeddings)}")
print(f"Dimension eines Embeddings: {len(embeddings[0])}")
print(f"Erste 5 Zahlen des ersten Embeddings: {embeddings[0][:5]}")

Anzahl Embeddings: 194
Dimension eines Embeddings: 1536
Erste 5 Zahlen des ersten Embeddings: [-0.024078369140625, -0.01018524169921875, -0.027191162109375, 0.023284912109375, -0.008575439453125]


## In Chroma speichern

Letzter Schritt: Chunks + Embeddings + Metadata in eine lokale Chroma-Datenbank schreiben — das ist unsere durchsuchbare Wissensbasis für den Agenten (Milestone 3).


In [8]:
%pip install chromadb

  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached uvicorn-0.52.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached numpy-2.5.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached onnxruntime-1.28.0-cp313-cp313-macosx_14_0_arm64.whl.metadata (5.5 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-macosx_10_12_universal2.whl.metadata (10 kB)
  Using cached typer-0.27.1-py3-none-any.whl.metadata (16 kB)
  Using cached kubernetes-36.0.3-py2.py3-none-any.whl.metadata (1.8 kB)
  Using cached pyyaml-6.0.3-cp313-cp

In [9]:
import chromadb

# PersistentClient speichert die DB als Dateien auf der Festplatte -- bleibt erhalten, auch nach Neustart des Notebooks
chroma_client = chromadb.PersistentClient(path="../data/chroma_db")

# Eine "Collection" ist wie eine Tabelle -- hier landen alle unsere Chunks
collection = chroma_client.get_or_create_collection(name="health_fitness_videos")

# Chroma braucht die Daten in getrennten Listen: IDs, Embeddings, Texte, Metadata
ids = [chunk["chunk_id"] for chunk in chunks_with_metadata]
metadatas = [
    {"video_id": chunk["video_id"], "title": chunk["title"], "start": chunk["start"], "end": chunk["end"]}
    for chunk in chunks_with_metadata
]
documents = [chunk["text"] for chunk in chunks_with_metadata]

collection.upsert(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
)

print(f"✅ {collection.count()} Chunks in Chroma gespeichert")

✅ 194 Chunks in Chroma gespeichert


## Migration: topic="health" nachträglich zu bestehenden Chunks hinzufügen

Die 194 Chunks aus dem MVP-Video wurden vor der Multi-Topic-Erweiterung gespeichert und
haben kein `topic`-Feld. Ergänzt es nachträglich, ohne die Chunks neu zu embedden.


In [1]:
import sys
sys.path.append("../backend")
from config import collection

# Alle bestehenden Chunks + ihre aktuelle Metadata holen
existing = collection.get(include=["metadatas"])
ids = existing["ids"]
metadatas = existing["metadatas"]

# topic="health" ergänzen, Rest der Metadata unverändert lassen
updated_metadatas = []
for m in metadatas:
    m["topic"] = "health"
    updated_metadatas.append(m)

collection.update(ids=ids, metadatas=updated_metadatas)
print(f"✅ {len(ids)} Chunks mit topic='health' aktualisiert")

✅ 194 Chunks mit topic='health' aktualisiert
